In [13]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

In [19]:
load_dotenv()

#1.a. DataLoader
#Load the transcript as Document via youtube-transcript-api 
import re
def manual_ytLoader(id_or_URL: str) -> Document:
    # Regex to handle regular URLs, shorts, youtu.be, and raw IDs
    pattern = r'(?:v=|\/shorts\/|\/embed\/|\/vi\/|youtu\.be\/|^\b)([a-zA-Z0-9_-]{11})'
    match = re.search(pattern, id_or_URL)
    
    if match:
        video_id = match.group(1)
    else:
        video_id = id_or_URL  # Fallback to the input string
    try:
        # 1. Initialize the API instance
        api_client = YouTubeTranscriptApi()
        
        # 2. Use .fetch() to get the transcript data
        raw_transcript = api_client.fetch(video_id)

        clean_text = " ".join(doc.text for doc in raw_transcript)

        if clean_text.strip():
            return Document(
                page_content=clean_text, 
                metadata={"source": f"<https://youtube.com/watch?v={video_id}>", "video_id": video_id}
            )
    except Exception as e:
        print(f"File Transcript cannot be loaded: {e} \n")

docs = manual_ytLoader("https://youtu.be/Gfr50f6ZBvo?si=Lk4Z8CWf7E1iv8em")

In [20]:
#1.b TextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

chunked_docs = splitter.split_documents([docs])

In [21]:
#1.cinitializing the vector_store

#first embedding model
embd = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")

vector_Store = FAISS.from_documents(
    documents= chunked_docs,
    embedding=embd
)

#2 Retrieval
retriever = vector_Store.as_retriever(search_kwargs = {"k":3})


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7692.41it/s]


In [22]:
#3. PromptTemplate Building

prompt = ChatPromptTemplate.from_template(
    """
    you are a helpful assistant. Help me answer the query : {query} , strictly with ONLY reference to
    the given Youtube transcript context : {context},
    If the context doesn't provide sufficient or relevant information simple say 'I don't know' or mention 
    that the video does not talk about the given query.
"""
)


In [23]:
# Defining a func to Merge all the recieved context docs into one single string
#to later make it a runnable and include in our LCEL

def contextMerger(docs : list[Document]) -> str:
    return " " \
    "\n\n".join(doc.page_content for doc in docs)

In [24]:
#4. Generation

query = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"

context = contextMerger(retriever.invoke(query))

final_prompt = prompt.invoke({"query": query, "context" : context })

In [29]:
#create a string output parser
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()

In [25]:
print(final_prompt)

messages=[HumanMessage(content="\n    you are a helpful assistant. Help me answer the query : is the topic of nuclear fusion discussed in this video? if yes then what was discussed , strictly with ONLY reference to\n    the given Youtube transcript context : in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we you know we get a fu

In [30]:
model = ChatGroq(model = "llama-3.3-70b-versatile")

res = parser.invoke(model.invoke(final_prompt))

In [32]:
print(res)

Yes, the topic of nuclear fusion is discussed in this video. 

The discussion revolves around the collaboration with EPFL in Switzerland, who have a test reactor that was made available for use. The team looked at the bottleneck problems in fusion and identified areas where their AI methods could be applied. 

One specific problem that was tackled was controlling and holding the plasma in specific shapes, which is a challenge in fusion. The team developed a controller that could contain the plasma and hold it in structure, and they were able to achieve a record amount of time in holding the plasma in specific shapes. 

The team also mentions that they are now looking to collaborate with fusion startups to tackle the next problem in the fusion area. 

Additionally, the challenges in fusion are mentioned, including physics, material science, and engineering challenges, such as building massive fusion reactors and containing the plasma.


In [33]:
# Building Chain

from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda

parall = RunnableParallel({
    "query" : RunnablePassthrough(),
    "context" : retriever | RunnableLambda(contextMerger)
})

forward_chain = prompt | model | parser

final_chain = parall | forward_chain

In [35]:
res = final_chain.invoke("Summarize the video in short")
print(res)

The video appears to be a conversation about the possibility of creating a deeper, simpler explanation of physics that goes beyond the standard model. The speaker mentions that this new explanation could provide glimpses into mysteries such as consciousness, life, and gravity. The conversation also touches on the idea of creating an artificial general intelligence (AGI) system that achieves human-level intelligence and beyond. If given the chance to ask the AGI system one question, the speaker would ask "what is the true nature of reality?" However, the video does not provide a clear answer to this question, instead, it shows the AGI system struggling to explain it.
